# 05_evaluation.ipynb

Load the trained aspect and sentiment models, run them on the preprocessed dataset, and report metrics and visualizations.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import sys
from pathlib import Path

# Ensure project root is in sys.path so 'src' is always importable
ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import pandas as pd
import joblib
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, classification_report
from src.predict import AspectSentimentPipeline

# Model paths
models_dir = ROOT_DIR / 'models'
aspect_vectorizer = joblib.load(models_dir / 'tfidf_vectorizer.joblib')
aspect_clf = joblib.load(models_dir / 'aspect_model.joblib')
sentiment_vectorizer = joblib.load(models_dir / 'tfidf_vectorizer_sentiment.joblib')
sentiment_clf = joblib.load(models_dir / 'sentiment_model.joblib')

# Load test set
test_df = pd.read_csv(ROOT_DIR / 'data' / 'processed' / 'reviews_preprocessed.csv')

# Predict aspects (binary matrix)
X_test = aspect_vectorizer.transform(test_df['review_processed'].fillna(''))
aspect_pred = aspect_clf.predict(X_test)
print('Aspect predictions shape:', aspect_pred.shape)

# Test full end-to-end pipeline on sample Hinglish reviews
pipeline = AspectSentimentPipeline(
    aspect_vectorizer=aspect_vectorizer,
    aspect_model=aspect_clf,
    sentiment_vectorizer=sentiment_vectorizer,
    sentiment_model=sentiment_clf
)

samples = [
    "Battery bahut accha hai but delivery late thi",
    "Price thoda high hai but quality mast hai",
    "Packaging bekar thi dabba tuta hua tha",
    "Paisa vasool product, charging speed is great"
]

print('\n--- Pipeline Sample Predictions ---')
for s in samples:
    print(f'Review: {s}')
    print('Output:', pipeline.predict(s))
    print('-' * 40)
